In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l3.2 Scaled dot-product

> 🎯 **Goal:** Build the attention formula `softmax(QK^T / sqrt(d)) V` by hand and prove it matches the engine primitive.

**Why this matters.** This one line is the entire computational core of attention. If you can reproduce it from `q`, `k`, and `v` using only `@` and `softmax`, you understand exactly what the library is doing under the hood, and you will never be surprised by a shape or a scaling bug again.

**The intuition.** We are scoring how well each query matches each key, softening those scores into weights, and using the weights to mix the values. The only new wrinkle is the `/ sqrt(d)` term: a volume knob that keeps the scores from getting so loud that the softmax stops listening to anything but the single loudest one.

**The mechanics.** Read the formula left to right:
- `Q K^T` is the score matrix. Row i, column j is `query_i . key_j`, the raw match between token i's query and token j's key. We compute it with `q @ k.transpose(-2, -1)`.
- `/ sqrt(d)` scales the scores down, where `d` is the head dimension (here 16). More on why below.
- `softmax(..., dim=-1)` turns each row of scores into a probability distribution over the keys.
- `... V` multiplies those weights by the value vectors, producing the weighted-average output.

We compute this by hand (`out_hand`) and compare it against `scaled_dot_product_attention`. If both `allclose` checks print `True`, our mental model and the engine agree to the last decimal.

In [ ]:
from torch.nn import functional as F
from pocketlm import scaled_dot_product_attention

torch.manual_seed(0)
q = torch.randn(1, 4, 16)
k = torch.randn(1, 4, 16)
v = torch.randn(1, 4, 16)
att = (q @ k.transpose(-2, -1)) / (16 ** 0.5)
w_hand = F.softmax(att, dim=-1)
out_hand = w_hand @ v
out, w = scaled_dot_product_attention(q, k, v)
print("weights match:", torch.allclose(w, w_hand))
print("output match: ", torch.allclose(out, out_hand))

**What just happened.** Both lines print `True`, which means our hand-built `softmax(QK^T / sqrt(d)) V` is bit-for-bit equivalent to the engine's `scaled_dot_product_attention`. There is no hidden magic in the primitive; it is exactly these four operations.

Why divide by `sqrt(d)`? A dot product of two `d`-dimensional vectors is a sum of `d` random products, so its variance grows with `d`. Larger `d` means larger and more spread-out scores. Feed very large scores into `softmax` and it saturates into a near one-hot spike: one weight is essentially 1 and the rest are essentially 0. A saturated softmax has almost-flat gradients, so training stalls (the vanishing-gradient problem). Dividing by `sqrt(d)` normalizes the variance back to roughly 1, keeping the distribution smooth and the gradients healthy.

⚠️ **Common pitfalls**
- Transposing the wrong axes. You want `k.transpose(-2, -1)` so the last two dims line up for the matmul; transposing the batch dim by mistake gives garbage or a shape error.
- Softmax over the wrong dimension. It must be `dim=-1` (over keys), so each query's weights sum to 1. Softmax over `dim=-2` would normalize down the columns, which is meaningless here.
- Scaling by `d` instead of `sqrt(d)`. The square root matters; dividing by `d` over-shrinks the scores and flattens attention.

🏋️ **Try it yourself.** Delete the `/ (16 ** 0.5)` from `att` and rerun. The hand-built and engine outputs will no longer match (the engine still scales), and if you also remove scaling from both, watch the weight rows drift toward one-hot as you increase the last dimension from 16 to 256.

In [ ]:
assert torch.allclose(w, w_hand)
assert torch.allclose(out, out_hand)